# 中证800 V50 Candidate Factor Expansion 实验

目的：从聚宽因子库新增一批小而精的候选因子，先做 OOS 因子审计，不直接进入模型。

本 notebook 分两段：

1. **因子扩展缓存**：读取现有月度样本，用 `feature_date` 从 `jqfactor.get_factor_values` 拉新因子，并保存增强 CSV。
2. **OOS 审计**：加载增强 CSV，做新增因子的单因子 / 因子组 / 交互审计。

纪律：

- 不导出 pkl
- 不训练实盘模型
- 不 early stopping
- 不 sample weight
- 因子方向和权重只用历史月份估计
- 聚宽 API 只用于补因子；审计阶段可完全离线运行


In [ ]:
import gc
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

BASE_DATA_PATH = "train_csi800_factor_v40_data_enhancement.csv"
OUT_DIR_ROOT = "csi800_ml_v50_candidate_factor_expansion_outputs"
os.makedirs(OUT_DIR_ROOT, exist_ok=True)

TARGET_COL = "alpha_1m"
TRAIN_START = "2019-01-01"
TRAIN_END = "2025-03-31"
MIN_HISTORY_MONTHS = 24
TOP_LIST = [10, 30]
WEIGHT_METHODS = ["equal", "ic_weight", "icir_weight"]
BASE_KEEP_COLS = ["stock", "rebalance_date", "feature_date", "next_date", TARGET_COL]

# Set False if you only want to analyze an existing ENRICHED_DATA_PATH cache.
FETCH_FACTORS_IF_CACHE_MISSING = True
FORCE_REFETCH_FACTORS = False

# OOM-safe defaults for JoinQuant notebooks. Increase only after a small group runs through.
FACTOR_CHUNK_SIZE = 1
STOCK_CHUNK_SIZE = 150
GC_EVERY_N_FETCH_DATES = 2
GC_EVERY_N_EVAL_MONTHS = 4

print("BASE_DATA_PATH =", BASE_DATA_PATH)
print("OUT_DIR_ROOT =", OUT_DIR_ROOT)


In [ ]:
ALL_EXPANSION_GROUPS = {
    "profit_quality_new": [
        "net_profit_ratio",
        "net_profit_to_total_operate_revenue_ttm",
        "cfo_to_ev",
        "net_operate_cash_flow_to_operate_income",
    ],
    "growth_improvement_new": [
        "DEGM",
        "operating_profit_growth_rate",
        "net_profit_growth_rate",
        "operating_revenue_growth_rate",
    ],
    "operation_efficiency_new": [
        "inventory_turnover_rate",
        "account_receivable_turnover_days",
        "inventory_turnover_days",
        "OperatingCycle",
    ],
    "long_quality_new": [
        "roe_ttm_8y",
        "roa_ttm_8y",
        "maximum_margin",
    ],
    "risk_volume_new": [
        "residual_volatility",
        "Skewness60",
        "turnover_volatility",
    ],
}

# Run one small group by default to avoid OOM. Change this list to test another group or a small pair.
ACTIVE_GROUP_NAMES = ["risk_volume_new"]

EXPANSION_GROUPS = {}
for group_name in ACTIVE_GROUP_NAMES:
    if group_name not in ALL_EXPANSION_GROUPS:
        raise ValueError("unknown group: " + str(group_name))
    EXPANSION_GROUPS[group_name] = list(ALL_EXPANSION_GROUPS[group_name])

PAIR_SPECS_ALL = [
    ("profit_quality_new", "growth_improvement_new"),
    ("profit_quality_new", "operation_efficiency_new"),
    ("long_quality_new", "risk_volume_new"),
    ("growth_improvement_new", "risk_volume_new"),
]
PAIR_SPECS = [(a, b) for (a, b) in PAIR_SPECS_ALL if a in EXPANSION_GROUPS and b in EXPANSION_GROUPS]

NEW_FACTOR_COLS = []
for group_name in EXPANSION_GROUPS:
    for f in EXPANSION_GROUPS[group_name]:
        if f not in NEW_FACTOR_COLS:
            NEW_FACTOR_COLS.append(f)

ACTIVE_GROUP_TAG = "__".join(ACTIVE_GROUP_NAMES)
OUT_DIR = os.path.join(OUT_DIR_ROOT, ACTIVE_GROUP_TAG)
os.makedirs(OUT_DIR, exist_ok=True)
ENRICHED_DATA_PATH = os.path.join(OUT_DIR, "csi800_v50_factor_expansion_dataset.csv")

print("active groups:", ACTIVE_GROUP_NAMES)
print("active new factors:", len(NEW_FACTOR_COLS), NEW_FACTOR_COLS)
print("pair specs:", PAIR_SPECS)
print("OUT_DIR =", OUT_DIR)
print("ENRICHED_DATA_PATH =", ENRICHED_DATA_PATH)


In [ ]:
def chunks(seq, size):
    seq = list(seq)
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def load_base_df(path, keep_cols=None):
    if not os.path.exists(path):
        raise IOError("base data not found: " + path)
    df = pd.read_csv(path)
    if "code" in df.columns and "stock" not in df.columns:
        df = df.rename(columns={"code": "stock"})
    for col in ["rebalance_date", "feature_date", "next_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.normalize()
    if TARGET_COL not in df.columns:
        raise ValueError("missing target: " + TARGET_COL)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    required = [c for c in ["stock", "rebalance_date", "feature_date", TARGET_COL] if c in df.columns]
    df = df.dropna(subset=required).copy()
    if keep_cols is not None:
        cols = [c for c in keep_cols if c in df.columns]
        extra = [c for c in NEW_FACTOR_COLS if c in df.columns and c not in cols]
        df = df[cols + extra].copy()
    return df


def fetch_factor_values_one_date(stock_list, factor_list, feature_date):
    try:
        from jqfactor import get_factor_values
    except Exception as err:
        raise RuntimeError("fetching new factors requires JoinQuant jqfactor runtime: " + str(err))

    date_str = pd.Timestamp(feature_date).strftime("%Y-%m-%d")
    rows = []
    for stock_chunk in chunks(stock_list, STOCK_CHUNK_SIZE):
        # One output row per stock-date. Factor chunks must be merged horizontally, not appended vertically.
        tmp = pd.DataFrame(index=stock_chunk)
        for factor_chunk in chunks(factor_list, FACTOR_CHUNK_SIZE):
            try:
                data = get_factor_values(
                    securities=stock_chunk,
                    factors=factor_chunk,
                    end_date=date_str,
                    count=1,
                )
            except Exception as err:
                print("factor fetch failed:", date_str, len(stock_chunk), factor_chunk, err)
                data = None
            for factor in factor_chunk:
                try:
                    if data is not None and factor in data:
                        tmp[factor] = data[factor].iloc[-1, :].reindex(stock_chunk)
                    else:
                        tmp[factor] = np.nan
                except Exception:
                    tmp[factor] = np.nan
            del data
        tmp["stock"] = tmp.index
        tmp["feature_date"] = pd.Timestamp(feature_date)
        rows.append(tmp.reset_index(drop=True))
        del tmp
        gc.collect()
    if not rows:
        out = pd.DataFrame({"stock": stock_list})
        out["feature_date"] = pd.Timestamp(feature_date)
        for f in factor_list:
            out[f] = np.nan
        return out
    out = pd.concat(rows, ignore_index=True, sort=False)
    del rows
    gc.collect()
    return out


def build_enriched_dataset(base_df):
    key_cols = ["stock", "feature_date"]
    compact_cols = [c for c in BASE_KEEP_COLS if c in base_df.columns]
    base_compact = base_df[compact_cols].copy()
    pairs = base_compact[key_cols].drop_duplicates().copy()
    out_parts = []
    feature_dates = sorted(pd.to_datetime(pairs["feature_date"].dropna().unique()))
    for i, dt in enumerate(feature_dates):
        stock_list = list(pairs[pairs["feature_date"] == dt]["stock"].drop_duplicates())
        print("fetch", i + 1, "/", len(feature_dates), pd.Timestamp(dt).strftime("%Y-%m-%d"), "stocks", len(stock_list))
        one = fetch_factor_values_one_date(stock_list, NEW_FACTOR_COLS, dt)
        out_parts.append(one)
        del one, stock_list
        if (i + 1) % GC_EVERY_N_FETCH_DATES == 0:
            gc.collect()
    factor_df = pd.concat(out_parts, ignore_index=True, sort=False) if out_parts else pd.DataFrame(columns=key_cols + NEW_FACTOR_COLS)
    del out_parts, pairs
    gc.collect()
    factor_df["feature_date"] = pd.to_datetime(factor_df["feature_date"]).dt.normalize()
    enriched = base_compact.merge(factor_df, on=key_cols, how="left", suffixes=("", "_newdup"), sort=False)
    del base_compact, factor_df
    gc.collect()
    dup_cols = [c for c in enriched.columns if c.endswith("_newdup")]
    if dup_cols:
        enriched = enriched.drop(columns=dup_cols)
    return enriched


In [ ]:
base_df = load_base_df(BASE_DATA_PATH, keep_cols=BASE_KEEP_COLS)
print("base compact:", base_df.shape, base_df["rebalance_date"].min(), base_df["rebalance_date"].max())

enriched_df = None
cache_valid = False
if os.path.exists(ENRICHED_DATA_PATH) and not FORCE_REFETCH_FACTORS:
    cached_df = load_base_df(ENRICHED_DATA_PATH, keep_cols=BASE_KEEP_COLS)
    dup_count = int(cached_df.duplicated(["stock", "feature_date"]).sum()) if set(["stock", "feature_date"]).issubset(cached_df.columns) else 0
    missing_factor_cols = [f for f in NEW_FACTOR_COLS if f not in cached_df.columns]
    if dup_count == 0 and len(missing_factor_cols) == 0:
        enriched_df = cached_df
        cache_valid = True
        print("loaded enriched cache:", ENRICHED_DATA_PATH, enriched_df.shape)
    else:
        print("invalid enriched cache, refetch required:", "dup_count=", dup_count, "missing_factor_cols=", missing_factor_cols)
        del cached_df
        gc.collect()

if enriched_df is None:
    if FETCH_FACTORS_IF_CACHE_MISSING:
        enriched_df = build_enriched_dataset(base_df)
        dup_count = int(enriched_df.duplicated(["stock", "feature_date"]).sum())
        if dup_count != 0:
            raise ValueError("enriched dataset has duplicated stock-feature_date rows: " + str(dup_count))
        enriched_df.to_csv(ENRICHED_DATA_PATH, index=False)
        print("saved enriched compact cache:", ENRICHED_DATA_PATH, enriched_df.shape)
    else:
        raise IOError("valid enriched cache missing and FETCH_FACTORS_IF_CACHE_MISSING=False: " + ENRICHED_DATA_PATH)

del base_df
gc.collect()

coverage_rows = []
for f in NEW_FACTOR_COLS:
    coverage_rows.append({"factor": f, "coverage": float(enriched_df[f].notnull().mean()) if f in enriched_df.columns else 0.0})
coverage_df = pd.DataFrame(coverage_rows).sort_values("coverage", ascending=False)
coverage_df.to_csv(os.path.join(OUT_DIR, "v50_new_factor_coverage.csv"), index=False)
print(coverage_df.to_string(index=False))


In [ ]:
def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def safe_stats(values):
    s = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"mean": np.nan, "std": np.nan, "ir": np.nan, "hit_rate": np.nan, "months": 0}
    std = s.std()
    return {
        "mean": float(s.mean()),
        "std": float(std) if not pd.isnull(std) else np.nan,
        "ir": float(s.mean() / std) if (not pd.isnull(std) and std > 0) else np.nan,
        "hit_rate": float((s > 0).mean()),
        "months": int(len(s)),
    }


def max_drawdown_from_returns(returns):
    s = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return np.nan
    nav = (1.0 + s).cumprod()
    return float((nav / nav.cummax() - 1.0).min())


def cross_section_rank_pct(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    out = pd.Series(index=s.index, dtype=float)
    valid = s.dropna()
    if len(valid) == 0:
        return out
    out.loc[valid.index] = valid.rank(method="average") / float(len(valid))
    return out


def calc_monthly_ic_table(hist_df, factors):
    rows = []
    for dt, g in hist_df.groupby("rebalance_date"):
        for factor in factors:
            if factor in g.columns:
                rows.append({"rebalance_date": dt, "factor": factor, "rank_ic": safe_rank_ic(g[factor], g[TARGET_COL])})
    return pd.DataFrame(rows)


def summarize_prior_ic(hist_df, factors):
    ic_df = calc_monthly_ic_table(hist_df, factors)
    rows = []
    for factor in factors:
        s = ic_df[ic_df["factor"] == factor]["rank_ic"] if not ic_df.empty else pd.Series(dtype=float)
        st = safe_stats(s)
        rows.append({"factor": factor, "ic_mean": st["mean"], "ic_std": st["std"], "ic_ir": st["ir"], "months": st["months"]})
    return pd.DataFrame(rows)


def get_factor_weights(prior_df, factors, method):
    factors = [f for f in factors if f in list(prior_df["factor"])]
    if len(factors) == 0:
        return {}
    m = prior_df.set_index("factor")
    raw = {}
    for f in factors:
        ic_mean = m.loc[f, "ic_mean"]
        ic_std = m.loc[f, "ic_std"]
        if pd.isnull(ic_mean):
            raw[f] = 0.0
        elif method == "equal":
            raw[f] = 1.0 if ic_mean >= 0 else -1.0
        elif method == "ic_weight":
            raw[f] = float(ic_mean)
        elif method == "icir_weight":
            raw[f] = float(ic_mean / ic_std) if (not pd.isnull(ic_std) and ic_std > 0) else 0.0
        else:
            raise ValueError("unknown method: " + str(method))
    denom = sum(abs(v) for v in raw.values())
    if denom <= 0:
        return {f: 0.0 for f in factors}
    return {f: raw[f] / denom for f in factors}


def score_group_month(month_df, factors, weights):
    parts = []
    for f in factors:
        if f not in month_df.columns or f not in weights:
            continue
        w = weights.get(f, 0.0)
        if w == 0:
            continue
        r = cross_section_rank_pct(month_df[f])
        parts.append((r - 0.5) * 2.0 * w)
    if len(parts) == 0:
        return pd.Series(index=month_df.index, dtype=float)
    return pd.concat(parts, axis=1).sum(axis=1)


def eval_topn(month_df, score_col, n):
    d = month_df[["stock", "rebalance_date", TARGET_COL, score_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(d) == 0:
        return None
    top = d.sort_values(score_col, ascending=False).head(min(n, len(d)))
    return {"mean_alpha": float(top[TARGET_COL].mean()), "count": int(len(top)), "targets": ",".join(list(top["stock"]))}


In [ ]:
work_df = enriched_df[(enriched_df["rebalance_date"] >= pd.Timestamp(TRAIN_START)) & (enriched_df["rebalance_date"] <= pd.Timestamp(TRAIN_END))].copy()
del enriched_df
gc.collect()

available_new_factors = [f for f in NEW_FACTOR_COLS if f in work_df.columns and work_df[f].notnull().mean() > 0.05]
missing_or_sparse = [f for f in NEW_FACTOR_COLS if f not in available_new_factors]
months = sorted(pd.to_datetime(work_df["rebalance_date"].dropna().unique()))
eval_months = months[MIN_HISTORY_MONTHS:]

print("work:", work_df.shape, work_df["rebalance_date"].min(), work_df["rebalance_date"].max(), "months", len(months))
print("eval months:", len(eval_months), eval_months[0] if eval_months else None, eval_months[-1] if eval_months else None)
print("available new factors:", available_new_factors)
print("missing or sparse:", missing_or_sparse)


In [ ]:
# OOS single-factor audit for newly added factors.
single_rows = []
single_month_rows = []
for factor_i, factor in enumerate(available_new_factors):
    for month_i, month in enumerate(eval_months):
        hist_df = work_df[work_df["rebalance_date"] < month]
        month_df = work_df[work_df["rebalance_date"] == month].copy()
        if hist_df["rebalance_date"].nunique() < MIN_HISTORY_MONTHS or month_df.empty:
            del month_df
            continue
        prior = summarize_prior_ic(hist_df, [factor])
        if prior.empty:
            del month_df, prior
            continue
        ic_mean = prior.iloc[0]["ic_mean"]
        direction = 1.0 if (pd.isnull(ic_mean) or ic_mean >= 0) else -1.0
        score_col = "score_" + factor
        month_df[score_col] = cross_section_rank_pct(month_df[factor]) * direction
        for n in TOP_LIST:
            r = eval_topn(month_df, score_col, n)
            if r is None:
                continue
            single_month_rows.append({
                "rebalance_date": month,
                "factor": factor,
                "topn": int(n),
                "mean_alpha": r["mean_alpha"],
                "count": r["count"],
                "direction": direction,
            })
        del month_df, prior
    if (factor_i + 1) % 4 == 0:
        print("single-factor processed:", factor_i + 1, "/", len(available_new_factors))
        gc.collect()

single_month_df = pd.DataFrame(single_month_rows)
if single_month_df.empty:
    single_summary_df = pd.DataFrame(columns=["factor", "topn", "months", "mean_monthly_alpha", "monthly_alpha_ir", "win_rate", "max_drawdown"])
else:
    for (factor, topn), g in single_month_df.groupby(["factor", "topn"]):
        st = safe_stats(g["mean_alpha"])
        single_rows.append({
            "factor": factor,
            "topn": int(topn),
            "months": st["months"],
            "mean_monthly_alpha": st["mean"],
            "monthly_alpha_ir": st["ir"],
            "win_rate": st["hit_rate"],
            "max_drawdown": max_drawdown_from_returns(g.sort_values("rebalance_date")["mean_alpha"]),
        })
    single_summary_df = pd.DataFrame(single_rows).sort_values(["topn", "mean_monthly_alpha"], ascending=[True, False])

single_month_df.to_csv(os.path.join(OUT_DIR, "v50_single_factor_oos_monthly.csv"), index=False)
single_summary_df.to_csv(os.path.join(OUT_DIR, "v50_single_factor_oos_summary.csv"), index=False)
print(single_summary_df[single_summary_df["topn"] == 10].head(30).to_string(index=False))


In [ ]:
# OOS group and pair audit for newly added factors.
def infer_strategy_type(score_col):
    if score_col.startswith("pair_add__"):
        return "pair_add"
    if score_col.startswith("pair_gate_"):
        return "pair_gate"
    return "group"


strategy_monthly_rows = []
factor_weight_rows = []
for i, month in enumerate(eval_months):
    hist_df = work_df[work_df["rebalance_date"] < month]
    month_df = work_df[work_df["rebalance_date"] == month].copy()
    if hist_df["rebalance_date"].nunique() < MIN_HISTORY_MONTHS or month_df.empty:
        del month_df
        continue
    prior = summarize_prior_ic(hist_df, available_new_factors)
    score_df = month_df[["stock", "rebalance_date", TARGET_COL]].copy()

    for group_name in sorted(EXPANSION_GROUPS):
        factors = [f for f in EXPANSION_GROUPS[group_name] if f in available_new_factors]
        for method in WEIGHT_METHODS:
            weights = get_factor_weights(prior, factors, method)
            score_col = "group__{}__{}".format(group_name, method)
            score_df[score_col] = score_group_month(month_df, factors, weights)
            for f, w in weights.items():
                factor_weight_rows.append({"rebalance_date": month, "strategy_name": score_col, "factor": f, "weight": w})

    for a, b in PAIR_SPECS:
        for method in WEIGHT_METHODS:
            col_a = "group__{}__{}".format(a, method)
            col_b = "group__{}__{}".format(b, method)
            if col_a not in score_df.columns or col_b not in score_df.columns:
                continue
            pair = a + "__" + b + "__" + method
            score_df["pair_add__" + pair] = (score_df[col_a] + score_df[col_b]) / 2.0
            gate_a = cross_section_rank_pct(score_df[col_a]) >= 0.60
            score_df["pair_gate_{}_then_{}__{}".format(a, b, method)] = score_df[col_b].where(gate_a)
            gate_b = cross_section_rank_pct(score_df[col_b]) >= 0.60
            score_df["pair_gate_{}_then_{}__{}".format(b, a, method)] = score_df[col_a].where(gate_b)

    score_cols = [c for c in score_df.columns if c not in ["stock", "rebalance_date", TARGET_COL]]
    for score_col in score_cols:
        stype = infer_strategy_type(score_col)
        for n in TOP_LIST:
            r = eval_topn(score_df, score_col, n)
            if r is None:
                continue
            strategy_monthly_rows.append({
                "rebalance_date": month,
                "strategy_name": score_col,
                "strategy_type": stype,
                "topn": int(n),
                "mean_alpha": r["mean_alpha"],
                "count": r["count"],
                "targets": r["targets"],
            })

    del month_df, prior, score_df
    if (i + 1) % GC_EVERY_N_EVAL_MONTHS == 0:
        print("processed eval months:", i + 1, "/", len(eval_months))
        gc.collect()

strategy_monthly_df = pd.DataFrame(strategy_monthly_rows)
factor_weight_df = pd.DataFrame(factor_weight_rows)
del strategy_monthly_rows, factor_weight_rows, work_df
gc.collect()
print("strategy_monthly_df:", strategy_monthly_df.shape)
print("factor_weight_df:", factor_weight_df.shape)


In [ ]:
def summarize_monthly(monthly_df):
    rows = []
    if monthly_df.empty:
        return pd.DataFrame(columns=["strategy_name", "strategy_type", "topn", "months", "mean_monthly_alpha", "monthly_alpha_ir", "win_rate", "max_drawdown"])
    for (strategy_name, topn), g in monthly_df.groupby(["strategy_name", "topn"]):
        st = safe_stats(g["mean_alpha"])
        rows.append({
            "strategy_name": strategy_name,
            "strategy_type": g["strategy_type"].iloc[0],
            "topn": int(topn),
            "months": st["months"],
            "mean_monthly_alpha": st["mean"],
            "monthly_alpha_ir": st["ir"],
            "win_rate": st["hit_rate"],
            "max_drawdown": max_drawdown_from_returns(g.sort_values("rebalance_date")["mean_alpha"]),
        })
    return pd.DataFrame(rows)


strategy_summary_df = summarize_monthly(strategy_monthly_df)
if not strategy_summary_df.empty:
    strategy_summary_df = strategy_summary_df.sort_values(["topn", "mean_monthly_alpha"], ascending=[True, False])

strategy_monthly_df.to_csv(os.path.join(OUT_DIR, "v50_strategy_monthly_alpha.csv"), index=False)
strategy_summary_df.to_csv(os.path.join(OUT_DIR, "v50_strategy_summary.csv"), index=False)
factor_weight_df.to_csv(os.path.join(OUT_DIR, "v50_factor_weights_oos.csv"), index=False)

print("top10 strategy summary:")
print(strategy_summary_df[strategy_summary_df["topn"] == 10].head(40).to_string(index=False))


In [ ]:
# Yearly stability for top candidates.
top_candidates = list(strategy_summary_df[strategy_summary_df["topn"] == 10].head(20)["strategy_name"])
year_rows = []
if not strategy_monthly_df.empty:
    tmp = strategy_monthly_df[strategy_monthly_df["strategy_name"].isin(top_candidates)].copy()
    tmp["year"] = pd.to_datetime(tmp["rebalance_date"]).dt.year
    for (strategy_name, topn, year), g in tmp.groupby(["strategy_name", "topn", "year"]):
        st = safe_stats(g["mean_alpha"])
        year_rows.append({
            "strategy_name": strategy_name,
            "strategy_type": g["strategy_type"].iloc[0],
            "topn": int(topn),
            "year": int(year),
            "months": st["months"],
            "mean_alpha": st["mean"],
            "ir": st["ir"],
            "win_rate": st["hit_rate"],
        })

yearly_summary_df = pd.DataFrame(year_rows).sort_values(["strategy_name", "topn", "year"])
yearly_summary_df.to_csv(os.path.join(OUT_DIR, "v50_yearly_summary_top_candidates.csv"), index=False)
print(yearly_summary_df.head(80).to_string(index=False))


In [ ]:
summary_xlsx = os.path.join(OUT_DIR, "v50_candidate_factor_expansion_summary.xlsx")
try:
    with pd.ExcelWriter(summary_xlsx) as writer:
        coverage_df.to_excel(writer, "coverage", index=False)
        single_summary_df.to_excel(writer, "single_factor_oos", index=False)
        strategy_summary_df.to_excel(writer, "strategy_summary", index=False)
        yearly_summary_df.to_excel(writer, "yearly_top_candidates", index=False)
        factor_weight_df.to_excel(writer, "factor_weights", index=False)
    print("saved xlsx:", summary_xlsx)
except Exception as err:
    print("xlsx export skipped:", err)

print("saved files:")
for fn in sorted(os.listdir(OUT_DIR)):
    print("  ", fn)


## 结论填写区

重点回填：

- 新增因子覆盖率是否足够？
- 哪些新单因子 OOS 有正 alpha？
- 哪些新增因子组 OOS 有价值？
- 是否有新因子能超过 V49 的 risk/momentum 主线？
- 如果没有，哪些因子可作为辅助过滤器？
- 下一步是否值得把新因子并入模型训练？

失败标准：

- 覆盖率低于 50% 且无明显 alpha：废弃。
- OOS top10 只在单一年份有效：标记 regime，不进入主模型。
- 不能超过或补充 V49 主线：暂不进入训练特征池。
